# 从零复现 Swin Transformer：窗口注意力、循环移位与层级特征

本 Notebook 不调用 `timm`、`torchvision.models`、`nn.MultiheadAttention` 或任何现成 Transformer。我们只用基础 PyTorch 手写 patch embedding、window partition/reverse、二维相对位置偏置、shifted-window mask、`SwinBlock.forward`、patch merging 与分类器。

重点不只是“网络能跑”：我们会用可逆性 oracle 检查窗口变换，用概率 oracle 证明循环边界没有泄漏，用相对坐标 oracle 检查 bias 索引，并明确奇数分辨率、非法 shape、梯度和发布制品的边界。全部离线、CPU 单线程；微型条纹任务只证明实现可学习，不代表 ImageNet 泛化。


## 1. 计算图、shape 与复杂度

```text
image [B,C,H,W]
  -> Conv2d(kernel=stride=P)                 -> tokens [B,H/P,W/P,D]
  -> W-MSA block + SW-MSA block              -> [B,h,w,D]
  -> 2×2 PatchMerging                        -> [B,ceil(h/2),ceil(w/2),2D]
  -> W-MSA block -> LayerNorm -> mean -> head -> [B,K]
```

全局注意力对 $N=HW$ 个 token 的注意力矩阵是 $O(N^2)$；窗口大小为 $M$ 时，窗口注意力约为 $O(NM^2)$。shifted window 不改变渐近复杂度，却让相邻 block 中原本分离的窗口交换信息。张量统一采用 `BHWC` 进入 block，因为窗口切分更直观；卷积入口仍是 `NCHW`。


In [ ]:
import warnings  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 计算并保存当前步骤的中间状态。

from copy import deepcopy  # 导入本单元所需的依赖。
from dataclasses import dataclass  # 导入本单元所需的依赖。
from hashlib import sha256  # 导入本单元所需的依赖。
from types import MappingProxyType  # 导入本单元所需的依赖。
import io  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED = 460728  # 计算并保存当前步骤的中间状态。
random.seed(SEED)  # 执行当前语句以推进本节示例。
np.random.seed(SEED)  # 执行当前语句以推进本节示例。
torch.manual_seed(SEED)  # 执行当前语句以推进本节示例。
torch.use_deterministic_algorithms(True)  # 执行当前语句以推进本节示例。
torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE = torch.device("cpu")  # 计算并保存当前步骤的中间状态。

assert DEVICE.type == "cpu"  # 用受控断言验证关键不变量。
assert torch.get_num_threads() == 1  # 用受控断言验证关键不变量。
assert torch.initial_seed() == SEED  # 用受控断言验证关键不变量。
print({"torch": torch.__version__, "device": str(DEVICE), "seed": SEED})  # 执行当前语句以推进本节示例。


## 2. Patch embedding：卷积就是共享的线性投影

对不重叠的 $P\times P$ patch，`Conv2d(kernel_size=P, stride=P)` 与“展开每个 patch 后乘同一矩阵”等价。这里选择**严格合同**：输入高宽必须能被 patch size 整除，不在模型内部静默裁剪或补零。生产系统若允许任意尺寸，应在预处理 recipe 中明确 pad 值和有效区域 mask。


In [ ]:
class PatchEmbedding(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_channels=3, embed_dim=24, patch_size=2):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if in_channels <= 0 or embed_dim <= 0 or patch_size <= 0:  # 按当前条件选择后续控制路径。
            raise ValueError("patch embedding dimensions must be positive")  # 遇到非法合同立即显式失败。
        self.in_channels = int(in_channels)  # 计算并保存当前步骤的中间状态。
        self.embed_dim = int(embed_dim)  # 计算并保存当前步骤的中间状态。
        self.patch_size = int(patch_size)  # 计算并保存当前步骤的中间状态。
        self.proj = nn.Conv2d(in_channels, embed_dim, patch_size, stride=patch_size)  # 计算并保存当前步骤的中间状态。
        self.norm = nn.LayerNorm(embed_dim)  # 计算并保存当前步骤的中间状态。

    def forward(self, images):  # 定义本节可复用的核心函数。
        if images.ndim != 4 or images.shape[1] != self.in_channels:  # 按当前条件选择后续控制路径。
            raise ValueError("expected NCHW with configured channels")  # 遇到非法合同立即显式失败。
        if not torch.is_floating_point(images) or not torch.isfinite(images).all():  # 按当前条件选择后续控制路径。
            raise ValueError("images must be finite floating point")  # 遇到非法合同立即显式失败。
        h, w = images.shape[-2:]  # 计算并保存当前步骤的中间状态。
        if h % self.patch_size or w % self.patch_size:  # 按当前条件选择后续控制路径。
            raise ValueError("height and width must be divisible by patch_size")  # 遇到非法合同立即显式失败。
        return self.norm(self.proj(images).permute(0, 2, 3, 1))  # 返回当前分支计算出的结果。

patcher = PatchEmbedding(3, 12, 2)  # 计算并保存当前步骤的中间状态。
patch_x = torch.randn(2, 3, 8, 10, requires_grad=True)  # 计算并保存当前步骤的中间状态。
patch_y = patcher(patch_x)  # 计算并保存当前步骤的中间状态。
assert patch_y.shape == (2, 4, 5, 12)  # 用受控断言验证关键不变量。
patch_y.square().mean().backward()  # 执行当前语句以推进本节示例。
assert patch_x.grad is not None and torch.isfinite(patch_x.grad).all()  # 用受控断言验证关键不变量。
assert float(patch_x.grad.abs().sum()) > 0  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    patcher(torch.randn(1, 3, 7, 8))  # 执行当前语句以推进本节示例。
    raise AssertionError("odd incompatible height must fail")  # 遇到非法合同立即显式失败。
except ValueError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。


## 3. Window partition/reverse：先证明是双射

`window_partition` 把 `[B,H,W,C]` 重排为 `[B*nW,M*M,C]`；`window_reverse` 必须精确恢复原张量。二者只有 `view/permute`，不应丢值或改变 token 次序。与其只检查 shape，更强的 oracle 是给每个位置唯一编号并逐元素比较往返结果。


In [ ]:
def window_partition(x, window_size):  # 定义本节可复用的核心函数。
    if x.ndim != 4 or window_size <= 0:  # 按当前条件选择后续控制路径。
        raise ValueError("x must be BHWC and window_size positive")  # 遇到非法合同立即显式失败。
    b, h, w, c = x.shape  # 计算并保存当前步骤的中间状态。
    if h % window_size or w % window_size:  # 按当前条件选择后续控制路径。
        raise ValueError("feature resolution must be divisible by window_size")  # 遇到非法合同立即显式失败。
    x = x.reshape(b, h // window_size, window_size, w // window_size, window_size, c)  # 计算并保存当前步骤的中间状态。
    return x.permute(0, 1, 3, 2, 4, 5).contiguous().reshape(-1, window_size ** 2, c)  # 返回当前分支计算出的结果。

def window_reverse(windows, window_size, height, width, batch_size):  # 定义本节可复用的核心函数。
    if windows.ndim != 3 or min(window_size, height, width, batch_size) <= 0:  # 按当前条件选择后续控制路径。
        raise ValueError("invalid window reverse arguments")  # 遇到非法合同立即显式失败。
    if height % window_size or width % window_size:  # 按当前条件选择后续控制路径。
        raise ValueError("height/width must be divisible by window_size")  # 遇到非法合同立即显式失败。
    expected = batch_size * (height // window_size) * (width // window_size)  # 计算并保存当前步骤的中间状态。
    if windows.shape[0] != expected or windows.shape[1] != window_size ** 2:  # 按当前条件选择后续控制路径。
        raise ValueError("window count or token count mismatch")  # 遇到非法合同立即显式失败。
    c = windows.shape[-1]  # 计算并保存当前步骤的中间状态。
    x = windows.reshape(batch_size, height // window_size, width // window_size,  # 计算并保存当前步骤的中间状态。
                        window_size, window_size, c)  # 执行当前语句以推进本节示例。
    return x.permute(0, 1, 3, 2, 4, 5).contiguous().reshape(batch_size, height, width, c)  # 返回当前分支计算出的结果。

numbered = torch.arange(2 * 8 * 12 * 3).reshape(2, 8, 12, 3)  # 计算并保存当前步骤的中间状态。
numbered_windows = window_partition(numbered, 4)  # 计算并保存当前步骤的中间状态。
numbered_back = window_reverse(numbered_windows, 4, 8, 12, 2)  # 计算并保存当前步骤的中间状态。
assert numbered_windows.shape == (12, 16, 3)  # 用受控断言验证关键不变量。
assert torch.equal(numbered, numbered_back)  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    window_partition(torch.zeros(1, 7, 8, 2), 4)  # 执行当前语句以推进本节示例。
    raise AssertionError("non-divisible feature must fail")  # 遇到非法合同立即显式失败。
except ValueError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。


## 4. 手写窗口注意力与二维相对位置偏置

每个 head 的注意力为

$$A=\operatorname{softmax}(QK^\top/\sqrt{d_h}+B_{\Delta h,\Delta w}+M).$$

窗口内两个位置的相对位移各落在 `[-M+1,M-1]`，所以 bias table 有 $(2M-1)^2$ 行。索引必须同时编码行差与列差；只按一维距离会错误地把“上方”和“左方”视为同一位置。下面的反向索引恒等式能抓出符号或 stride 写错。


In [ ]:
class WindowAttention(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, dim, window_size, num_heads):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if dim <= 0 or window_size <= 0 or num_heads <= 0 or dim % num_heads:  # 按当前条件选择后续控制路径。
            raise ValueError("dim must be divisible by positive num_heads")  # 遇到非法合同立即显式失败。
        self.dim, self.window_size, self.num_heads = int(dim), int(window_size), int(num_heads)  # 计算并保存当前步骤的中间状态。
        self.head_dim = dim // num_heads  # 计算并保存当前步骤的中间状态。
        self.scale = self.head_dim ** -0.5  # 计算并保存当前步骤的中间状态。
        self.qkv = nn.Linear(dim, 3 * dim)  # 计算并保存当前步骤的中间状态。
        self.proj = nn.Linear(dim, dim)  # 计算并保存当前步骤的中间状态。
        table_size = (2 * window_size - 1) ** 2  # 计算并保存当前步骤的中间状态。
        self.relative_position_bias_table = nn.Parameter(torch.zeros(table_size, num_heads))  # 计算并保存当前步骤的中间状态。
        nn.init.trunc_normal_(self.relative_position_bias_table, std=0.02)  # 计算并保存当前步骤的中间状态。

        coords = torch.stack(torch.meshgrid(torch.arange(window_size), torch.arange(window_size), indexing="ij"))  # 计算并保存当前步骤的中间状态。
        flat = coords.flatten(1)  # 计算并保存当前步骤的中间状态。
        relative = flat[:, :, None] - flat[:, None, :]  # 计算并保存当前步骤的中间状态。
        relative = relative.permute(1, 2, 0).contiguous()  # 计算并保存当前步骤的中间状态。
        relative[:, :, 0] += window_size - 1  # 计算并保存当前步骤的中间状态。
        relative[:, :, 1] += window_size - 1  # 计算并保存当前步骤的中间状态。
        relative[:, :, 0] *= 2 * window_size - 1  # 计算并保存当前步骤的中间状态。
        self.register_buffer("relative_position_index", relative.sum(-1).long())  # 执行当前语句以推进本节示例。

    def forward(self, windows, attention_mask=None, return_attention=False):  # 定义本节可复用的核心函数。
        if windows.ndim != 3 or windows.shape[1] != self.window_size ** 2 or windows.shape[2] != self.dim:  # 按当前条件选择后续控制路径。
            raise ValueError("windows must be [BnW, M*M, dim]")  # 遇到非法合同立即显式失败。
        if not torch.isfinite(windows).all():  # 按当前条件选择后续控制路径。
            raise ValueError("window tokens must be finite")  # 遇到非法合同立即显式失败。
        bnw, n, _ = windows.shape  # 计算并保存当前步骤的中间状态。
        qkv = self.qkv(windows).reshape(bnw, n, 3, self.num_heads, self.head_dim)  # 计算并保存当前步骤的中间状态。
        q, k, v = qkv.permute(2, 0, 3, 1, 4)  # 计算并保存当前步骤的中间状态。
        scores = (q @ k.transpose(-2, -1)) * self.scale  # 计算并保存当前步骤的中间状态。
        bias = self.relative_position_bias_table[self.relative_position_index.reshape(-1)]  # 计算并保存当前步骤的中间状态。
        bias = bias.reshape(n, n, self.num_heads).permute(2, 0, 1)  # 计算并保存当前步骤的中间状态。
        scores = scores + bias.unsqueeze(0)  # 计算并保存当前步骤的中间状态。
        if attention_mask is not None:  # 按当前条件选择后续控制路径。
            if attention_mask.ndim != 3 or attention_mask.shape[0] == 0 or attention_mask.shape[1:] != (n, n):  # 按当前条件选择后续控制路径。
                raise ValueError("attention_mask must be nonempty [nW, M*M, M*M]")  # 遇到非法合同立即显式失败。
            if not torch.is_floating_point(attention_mask) or not torch.isfinite(attention_mask).all():  # 按当前条件选择后续控制路径。
                raise ValueError("attention_mask must be finite floating point")  # 遇到非法合同立即显式失败。
            if attention_mask.device != scores.device or attention_mask.dtype != scores.dtype:  # 按当前条件选择后续控制路径。
                raise ValueError("attention_mask must match attention score dtype/device")  # 遇到非法合同立即显式失败。
            # 0 表示允许；小于等于 -20 的值才足以作为 float32 安全屏蔽。
            allowed = (attention_mask == 0) | (attention_mask <= -20)  # 计算并保存当前步骤的中间状态。
            if not bool(allowed.all()):  # 按当前条件选择后续控制路径。
                raise ValueError("attention_mask values must be 0 or a safe negative blocker <= -20")  # 遇到非法合同立即显式失败。
            nwin = attention_mask.shape[0]  # 计算并保存当前步骤的中间状态。
            if bnw % nwin:  # 按当前条件选择后续控制路径。
                raise ValueError("batch-window count is incompatible with mask")  # 遇到非法合同立即显式失败。
            scores = scores.reshape(bnw // nwin, nwin, self.num_heads, n, n)  # 计算并保存当前步骤的中间状态。
            scores = scores + attention_mask[None, :, None]  # 计算并保存当前步骤的中间状态。
            scores = scores.reshape(bnw, self.num_heads, n, n)  # 计算并保存当前步骤的中间状态。
        probs = scores.softmax(dim=-1)  # 计算并保存当前步骤的中间状态。
        out = (probs @ v).transpose(1, 2).reshape(bnw, n, self.dim)  # 计算并保存当前步骤的中间状态。
        out = self.proj(out)  # 计算并保存当前步骤的中间状态。
        return (out, probs) if return_attention else out  # 返回当前分支计算出的结果。

attn_probe = WindowAttention(16, 4, 4)  # 计算并保存当前步骤的中间状态。
idx = attn_probe.relative_position_index  # 计算并保存当前步骤的中间状态。
center = (4 - 1) * (2 * 4 - 1) + (4 - 1)  # 计算并保存当前步骤的中间状态。
assert idx.shape == (16, 16)  # 用受控断言验证关键不变量。
assert torch.equal(torch.diag(idx), torch.full((16,), center, dtype=torch.long))  # 用受控断言验证关键不变量。
assert torch.equal(idx + idx.T, torch.full_like(idx, 2 * center))  # 用受控断言验证关键不变量。
assert int(idx.min()) == 0 and int(idx.max()) == (2 * 4 - 1) ** 2 - 1  # 用受控断言验证关键不变量。
attn_input = torch.randn(3, 16, 16, requires_grad=True)  # 计算并保存当前步骤的中间状态。
attn_output = attn_probe(attn_input)  # 计算并保存当前步骤的中间状态。
attn_output.sum().backward()  # 执行当前语句以推进本节示例。
assert attn_output.shape == attn_input.shape  # 用受控断言验证关键不变量。
assert attn_input.grad is not None and torch.isfinite(attn_input.grad).all()  # 用受控断言验证关键不变量。
assert attn_probe.scale == 0.5  # head_dim=4 的 1/sqrt(d_h) 数值 oracle
assert int(idx[0, 1]) == 23 and int(idx[0, 4]) == 17  # 用受控断言验证关键不变量。

valid_mask46 = torch.zeros(1, 16, 16)  # 计算并保存当前步骤的中间状态。
valid_mask46[:, 0, 1] = -100.0  # 计算并保存当前步骤的中间状态。
valid_out46, valid_probs46 = attn_probe(attn_input.detach(), valid_mask46, return_attention=True)  # 计算并保存当前步骤的中间状态。
assert torch.isfinite(valid_out46).all()  # 用受控断言验证关键不变量。
assert float(valid_probs46[:, :, 0, 1].max()) < 1e-40  # 用受控断言验证关键不变量。

bad_masks46 = [  # 计算并保存当前步骤的中间状态。
    torch.full((1, 16, 16), float("nan")),  # 执行当前语句以推进本节示例。
    torch.full((1, 16, 16), 0.25),  # 执行当前语句以推进本节示例。
    torch.zeros(1, 16, 16, dtype=torch.float64),  # 计算并保存当前步骤的中间状态。
]  # 执行当前语句以推进本节示例。
for bad_mask46 in bad_masks46:  # 遍历输入元素以累积或检查结果。
    try:  # 尝试执行可能失败的受控操作。
        attn_probe(attn_input.detach(), bad_mask46)  # 执行当前语句以推进本节示例。
        raise AssertionError("非法 attention mask 未 fail-closed")  # 遇到非法合同立即显式失败。
    except ValueError as exc:  # 捕获预期异常并验证失败分支。
        assert "attention_mask" in str(exc)  # 用受控断言验证关键不变量。


## 5. Shifted-window mask：允许邻窗交流，但禁止环绕边界“穿越”

SW-MSA 先把特征循环左移/上移，再按固定窗口切分。循环移位会把图像最右侧卷到最左侧；这些并非真实邻居，必须用 region mask 阻断。注意：mask **不能**简单按原窗口 ID 分组，否则所有跨窗口连接都会被禁掉，SW-MSA 就退化了。

标准做法把每个轴切为 `[0:-M]、[-M:-s]、[-s:]` 三段，二维组合成九个 region。窗口内不同 region 的 pair 加 `-100`；相同 region 加 0。


In [ ]:
def build_shifted_window_mask(height, width, window_size, shift_size, device=None):  # 定义本节可复用的核心函数。
    if min(height, width, window_size) <= 0 or not 0 < shift_size < window_size:  # 按当前条件选择后续控制路径。
        raise ValueError("require 0 < shift_size < window_size")  # 遇到非法合同立即显式失败。
    if height % window_size or width % window_size:  # 按当前条件选择后续控制路径。
        raise ValueError("resolution must be divisible by window_size")  # 遇到非法合同立即显式失败。
    region = torch.zeros((1, height, width, 1), device=device)  # 计算并保存当前步骤的中间状态。
    h_slices = (slice(0, -window_size), slice(-window_size, -shift_size), slice(-shift_size, None))  # 计算并保存当前步骤的中间状态。
    w_slices = (slice(0, -window_size), slice(-window_size, -shift_size), slice(-shift_size, None))  # 计算并保存当前步骤的中间状态。
    count = 0  # 计算并保存当前步骤的中间状态。
    for hs in h_slices:  # 遍历输入元素以累积或检查结果。
        for ws in w_slices:  # 遍历输入元素以累积或检查结果。
            region[:, hs, ws, :] = count  # 计算并保存当前步骤的中间状态。
            count += 1  # 计算并保存当前步骤的中间状态。
    region_windows = window_partition(region, window_size).squeeze(-1)  # 计算并保存当前步骤的中间状态。
    differences = region_windows[:, :, None] - region_windows[:, None, :]  # 计算并保存当前步骤的中间状态。
    return differences.ne(0).to(torch.float32) * -100.0  # 返回当前分支计算出的结果。

shift_mask = build_shifted_window_mask(8, 8, 4, 2)  # 计算并保存当前步骤的中间状态。
assert shift_mask.shape == (4, 16, 16)  # 用受控断言验证关键不变量。
assert set(shift_mask.unique().tolist()) == {-100.0, -0.0}  # 用受控断言验证关键不变量。
assert torch.equal(torch.diagonal(shift_mask, dim1=-2, dim2=-1), torch.zeros(4, 16))  # 用受控断言验证关键不变量。
assert bool((shift_mask == -100).any())  # 用受控断言验证关键不变量。
assert bool(((shift_mask == 0) & ~torch.eye(16, dtype=torch.bool)[None]).any())  # 用受控断言验证关键不变量。

# 概率 oracle：被 mask 的 pair 在 softmax 后必须数值上为零，而非“只是较小”。
zero_scores = torch.zeros_like(shift_mask)  # 计算并保存当前步骤的中间状态。
masked_probs = (zero_scores + shift_mask).softmax(-1)  # 计算并保存当前步骤的中间状态。
assert float(masked_probs[shift_mask == -100].max()) < 1e-40  # 用受控断言验证关键不变量。
assert torch.allclose(masked_probs.sum(-1), torch.ones_like(masked_probs.sum(-1)))  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    build_shifted_window_mask(7, 8, 4, 2)  # 执行当前语句以推进本节示例。
    raise AssertionError("illegal shifted resolution must fail")  # 遇到非法合同立即显式失败。
except ValueError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。


## 6. Swin block：Pre-Norm、残差与 cyclic shift

一个 block 先 `LayerNorm -> (S)W-MSA -> residual`，再 `LayerNorm -> MLP -> residual`。shift block 的顺序是：负向 roll、窗口切分、带 mask 注意力、窗口恢复、正向 roll。`shift_size=0` 的普通窗口 block 不使用 mask。DropPath 为教学简洁省略，不影响核心算子定义。


In [ ]:
class FeedForward(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, dim, hidden_dim):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.fc1 = nn.Linear(dim, hidden_dim)  # 计算并保存当前步骤的中间状态。
        self.fc2 = nn.Linear(hidden_dim, dim)  # 计算并保存当前步骤的中间状态。

    def forward(self, x):  # 定义本节可复用的核心函数。
        return self.fc2(F.gelu(self.fc1(x)))  # 返回当前分支计算出的结果。


class SwinBlock(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, dim, input_resolution, num_heads, window_size=4, shift_size=0, mlp_ratio=2):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        h, w = input_resolution  # 计算并保存当前步骤的中间状态。
        if h % window_size or w % window_size or not 0 <= shift_size < window_size:  # 按当前条件选择后续控制路径。
            raise ValueError("resolution/shift is incompatible with window")  # 遇到非法合同立即显式失败。
        self.dim = int(dim)  # 计算并保存当前步骤的中间状态。
        self.input_resolution = (int(h), int(w))  # 计算并保存当前步骤的中间状态。
        self.window_size, self.shift_size = int(window_size), int(shift_size)  # 计算并保存当前步骤的中间状态。
        self.norm1 = nn.LayerNorm(dim)  # 计算并保存当前步骤的中间状态。
        self.attention = WindowAttention(dim, window_size, num_heads)  # 计算并保存当前步骤的中间状态。
        self.norm2 = nn.LayerNorm(dim)  # 计算并保存当前步骤的中间状态。
        self.mlp = FeedForward(dim, dim * mlp_ratio)  # 计算并保存当前步骤的中间状态。
        if shift_size:  # 按当前条件选择后续控制路径。
            mask = build_shifted_window_mask(h, w, window_size, shift_size)  # 计算并保存当前步骤的中间状态。
        else:  # 处理前置条件不成立的分支。
            mask = torch.empty(0)  # 计算并保存当前步骤的中间状态。
        self.register_buffer("attention_mask", mask)  # 执行当前语句以推进本节示例。

    def forward(self, x):  # 定义本节可复用的核心函数。
        if x.ndim != 4 or tuple(x.shape[1:3]) != self.input_resolution or x.shape[-1] != self.dim:  # 按当前条件选择后续控制路径。
            raise ValueError("SwinBlock received an incompatible BHWC tensor")  # 遇到非法合同立即显式失败。
        shortcut = x  # 计算并保存当前步骤的中间状态。
        y = self.norm1(x)  # 计算并保存当前步骤的中间状态。
        if self.shift_size:  # 按当前条件选择后续控制路径。
            y = torch.roll(y, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))  # 计算并保存当前步骤的中间状态。
        windows = window_partition(y, self.window_size)  # 计算并保存当前步骤的中间状态。
        mask = self.attention_mask if self.shift_size else None  # 计算并保存当前步骤的中间状态。
        windows = self.attention(windows, mask)  # 计算并保存当前步骤的中间状态。
        y = window_reverse(windows, self.window_size, *self.input_resolution, x.shape[0])  # 计算并保存当前步骤的中间状态。
        if self.shift_size:  # 按当前条件选择后续控制路径。
            y = torch.roll(y, shifts=(self.shift_size, self.shift_size), dims=(1, 2))  # 计算并保存当前步骤的中间状态。
        x = shortcut + y  # 计算并保存当前步骤的中间状态。
        return x + self.mlp(self.norm2(x))  # 返回当前分支计算出的结果。

regular_block = SwinBlock(16, (8, 8), 4, 4, 0)  # 计算并保存当前步骤的中间状态。
shifted_block = SwinBlock(16, (8, 8), 4, 4, 2)  # 计算并保存当前步骤的中间状态。
block_x = torch.randn(2, 8, 8, 16, requires_grad=True)  # 计算并保存当前步骤的中间状态。
block_y = shifted_block(regular_block(block_x))  # 计算并保存当前步骤的中间状态。
block_y.mean().backward()  # 执行当前语句以推进本节示例。
assert block_y.shape == block_x.shape  # 用受控断言验证关键不变量。
assert block_x.grad is not None and torch.isfinite(block_x.grad).all()  # 用受控断言验证关键不变量。
assert float(block_x.grad.norm()) > 0  # 用受控断言验证关键不变量。
assert shifted_block.attention_mask.numel() == 4 * 16 * 16  # 用受控断言验证关键不变量。


## 7. Patch merging：显式处理奇数分辨率

层级视觉模型需要降采样。`PatchMerging` 取 `(偶行偶列、奇行偶列、偶行奇列、奇行奇列)` 四组 token，在通道维拼成 `4C`，归一化后投影到 `2C`。

这里与 patch embedding 的严格策略不同：中间特征可能因真实输入长宽而成为奇数，所以明确在**右侧/底部补零**并返回 `ceil(H/2),ceil(W/2)`。这不是悄悄行为：artifact 的 coordinate recipe 会绑定这一约定。


In [ ]:
class PatchMerging(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, dim):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if dim <= 0:  # 按当前条件选择后续控制路径。
            raise ValueError("dim must be positive")  # 遇到非法合同立即显式失败。
        self.dim = int(dim)  # 计算并保存当前步骤的中间状态。
        self.norm = nn.LayerNorm(4 * dim)  # 计算并保存当前步骤的中间状态。
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)  # 计算并保存当前步骤的中间状态。

    def forward(self, x):  # 定义本节可复用的核心函数。
        if x.ndim != 4 or x.shape[-1] != self.dim:  # 按当前条件选择后续控制路径。
            raise ValueError("PatchMerging expects BHWC with configured dim")  # 遇到非法合同立即显式失败。
        b, h, w, c = x.shape  # 计算并保存当前步骤的中间状态。
        if h % 2 or w % 2:  # 按当前条件选择后续控制路径。
            x = F.pad(x, (0, 0, 0, w % 2, 0, h % 2))  # 计算并保存当前步骤的中间状态。
        x00, x10 = x[:, 0::2, 0::2], x[:, 1::2, 0::2]  # 计算并保存当前步骤的中间状态。
        x01, x11 = x[:, 0::2, 1::2], x[:, 1::2, 1::2]  # 计算并保存当前步骤的中间状态。
        merged = torch.cat([x00, x10, x01, x11], dim=-1)  # 计算并保存当前步骤的中间状态。
        return self.reduction(self.norm(merged))  # 返回当前分支计算出的结果。

merger = PatchMerging(6)  # 计算并保存当前步骤的中间状态。
even_merged = merger(torch.randn(2, 8, 6, 6))  # 计算并保存当前步骤的中间状态。
odd_merged = merger(torch.randn(2, 5, 7, 6))  # 计算并保存当前步骤的中间状态。
assert even_merged.shape == (2, 4, 3, 12)  # 用受控断言验证关键不变量。
assert odd_merged.shape == (2, 3, 4, 12)  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    merger(torch.randn(1, 5, 7, 5))  # 执行当前语句以推进本节示例。
    raise AssertionError("wrong channel count must fail")  # 遇到非法合同立即显式失败。
except ValueError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。


## 8. 组装 Tiny Swin 分类器

教学模型使用 `16×16` 输入、`2×2` patch、两层 stage。第一 stage 的两个 block 分别使用 W-MSA 与 SW-MSA；合并后通道翻倍，再做一个普通窗口 block。真实 Swin-T 有更多 block、stochastic depth 和精心调参，这里保留核心 forward 数据流。


In [ ]:
class TinySwinClassifier(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, image_size=16, in_channels=1, num_classes=3, embed_dim=16):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if image_size != 16:  # 按当前条件选择后续控制路径。
            raise ValueError("this audited teaching configuration binds image_size=16")  # 遇到非法合同立即显式失败。
        self.image_size, self.in_channels, self.num_classes = image_size, in_channels, num_classes  # 计算并保存当前步骤的中间状态。
        self.patch_embed = PatchEmbedding(in_channels, embed_dim, patch_size=2)  # 计算并保存当前步骤的中间状态。
        self.stage1 = nn.ModuleList([  # 计算并保存当前步骤的中间状态。
            SwinBlock(embed_dim, (8, 8), 4, 4, 0),  # 执行当前语句以推进本节示例。
            SwinBlock(embed_dim, (8, 8), 4, 4, 2),  # 执行当前语句以推进本节示例。
        ])  # 执行当前语句以推进本节示例。
        self.merge = PatchMerging(embed_dim)  # 计算并保存当前步骤的中间状态。
        self.stage2 = SwinBlock(2 * embed_dim, (4, 4), 4, 2, 0)  # 计算并保存当前步骤的中间状态。
        self.norm = nn.LayerNorm(2 * embed_dim)  # 计算并保存当前步骤的中间状态。
        self.head = nn.Linear(2 * embed_dim, num_classes)  # 计算并保存当前步骤的中间状态。

    def forward(self, images):  # 定义本节可复用的核心函数。
        if images.ndim != 4 or images.shape[1:] != (self.in_channels, self.image_size, self.image_size):  # 按当前条件选择后续控制路径。
            raise ValueError("classifier expects the published NCHW shape")  # 遇到非法合同立即显式失败。
        x = self.patch_embed(images)  # 计算并保存当前步骤的中间状态。
        for block in self.stage1:  # 遍历输入元素以累积或检查结果。
            x = block(x)  # 计算并保存当前步骤的中间状态。
        x = self.merge(x)  # 计算并保存当前步骤的中间状态。
        x = self.stage2(x)  # 计算并保存当前步骤的中间状态。
        return self.head(self.norm(x).mean(dim=(1, 2)))  # 返回当前分支计算出的结果。

swin_model = TinySwinClassifier().to(DEVICE)  # 计算并保存当前步骤的中间状态。
swin_logits = swin_model(torch.randn(4, 1, 16, 16))  # 计算并保存当前步骤的中间状态。
assert swin_logits.shape == (4, 3)  # 用受控断言验证关键不变量。
assert torch.isfinite(swin_logits).all()  # 用受控断言验证关键不变量。
parameter_count = sum(p.numel() for p in swin_model.parameters())  # 计算并保存当前步骤的中间状态。
assert 10_000 < parameter_count < 100_000  # 用受控断言验证关键不变量。


## 9. 受控训练：只验证端到端梯度，不冒充视觉泛化

我们合成三类极简单图案：竖条、横条和十字，并固定 train/test seed。目标是检查 patch、attention、merge、head 的梯度能共同降低损失。因为分布是人工设计且样本极少，即使准确率很高，也不能外推到自然图像。


In [ ]:
def make_swin_patterns(count, seed):  # 定义本节可复用的核心函数。
    generator = torch.Generator().manual_seed(seed)  # 计算并保存当前步骤的中间状态。
    labels = torch.arange(count) % 3  # 计算并保存当前步骤的中间状态。
    images = 0.04 * torch.randn(count, 1, 16, 16, generator=generator)  # 计算并保存当前步骤的中间状态。
    for i, label in enumerate(labels.tolist()):  # 遍历输入元素以累积或检查结果。
        offset = int(torch.randint(5, 11, (1,), generator=generator))  # 计算并保存当前步骤的中间状态。
        if label in (0, 2):  # 按当前条件选择后续控制路径。
            images[i, :, :, offset-1:offset+1] += 1.0  # 计算并保存当前步骤的中间状态。
        if label in (1, 2):  # 按当前条件选择后续控制路径。
            images[i, :, offset-1:offset+1, :] += 1.0  # 计算并保存当前步骤的中间状态。
    return images.clamp(0, 1), labels.long()  # 返回当前分支计算出的结果。

train_images46, train_labels46 = make_swin_patterns(36, SEED + 1)  # 计算并保存当前步骤的中间状态。
test_images46, test_labels46 = make_swin_patterns(18, SEED + 2)  # 计算并保存当前步骤的中间状态。
assert not torch.equal(train_images46[:18], test_images46)  # 用受控断言验证关键不变量。
assert set(train_labels46.tolist()) == {0, 1, 2}  # 用受控断言验证关键不变量。

optimizer = torch.optim.AdamW(swin_model.parameters(), lr=5e-3, weight_decay=1e-4)  # 计算并保存当前步骤的中间状态。
loss_trace46 = []  # 计算并保存当前步骤的中间状态。
swin_model.train()  # 执行当前语句以推进本节示例。
for step in range(38):  # 遍历输入元素以累积或检查结果。
    optimizer.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
    loss = F.cross_entropy(swin_model(train_images46), train_labels46)  # 计算并保存当前步骤的中间状态。
    loss.backward()  # 执行当前语句以推进本节示例。
    optimizer.step()  # 执行当前语句以推进本节示例。
    loss_trace46.append(float(loss.detach()))  # 执行当前语句以推进本节示例。

swin_model.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    test_accuracy46 = float((swin_model(test_images46).argmax(-1) == test_labels46).float().mean())  # 计算并保存当前步骤的中间状态。
assert loss_trace46[-1] < 0.25 * loss_trace46[0]  # 用受控断言验证关键不变量。
assert test_accuracy46 >= 0.94  # 用受控断言验证关键不变量。
assert all(math.isfinite(v) for v in loss_trace46)  # 用受控断言验证关键不变量。
print({"initial_loss": round(loss_trace46[0], 4), "final_loss": round(loss_trace46[-1], 4),  # 执行当前语句以推进本节示例。
       "controlled_test_accuracy": test_accuracy46})  # 执行当前语句以推进本节示例。


## 10. 发布制品：认证完整 package，并返回不可变语义 bundle

攻击者若能整体替换权重、配置并重算 package 内所有 hash，“hash 都一致”仍不能证明这是发布方批准的模型。因此 loader 除内部一致性外，还查询 package **之外**的只读 publisher registry：`artifact_id -> expected package digest`。

canonical state digest 按排序后的参数名绑定 `key + dtype + shape + raw bytes`。整体 digest 还覆盖 architecture、attention/mask、数据 split、像素预处理、patch/merge 坐标与标签语义。loader 会逐项与代码支持的合同交叉校验，不能只依赖“已签名但内部互相矛盾”的 metadata。

返回值是 `PublishedSwin`，而非丢失标签和预处理语义的裸 `nn.Module`。其中 metadata 递归冻结；`predict` 同时执行输入范围和 dtype 合同。


In [ ]:
def state_digest46(state):  # 定义本节可复用的核心函数。
    digest = sha256()  # 计算并保存当前步骤的中间状态。
    for key in sorted(state):  # 遍历输入元素以累积或检查结果。
        tensor = state[key].detach().cpu().contiguous()  # 计算并保存当前步骤的中间状态。
        digest.update(key.encode())  # 执行当前语句以推进本节示例。
        digest.update(str(tensor.dtype).encode())  # 执行当前语句以推进本节示例。
        digest.update(json.dumps(list(tensor.shape)).encode())  # 执行当前语句以推进本节示例。
        digest.update(tensor.numpy().tobytes())  # 执行当前语句以推进本节示例。
    return digest.hexdigest()  # 返回当前分支计算出的结果。

def tensor_digest46(*tensors):  # 定义本节可复用的核心函数。
    digest = sha256()  # 计算并保存当前步骤的中间状态。
    for tensor in tensors:  # 遍历输入元素以累积或检查结果。
        value = tensor.detach().cpu().contiguous()  # 计算并保存当前步骤的中间状态。
        digest.update(str(value.dtype).encode())  # 执行当前语句以推进本节示例。
        digest.update(json.dumps(list(value.shape)).encode())  # 执行当前语句以推进本节示例。
        digest.update(value.numpy().tobytes())  # 执行当前语句以推进本节示例。
    return digest.hexdigest()  # 返回当前分支计算出的结果。

def package_digest46(package):  # 定义本节可复用的核心函数。
    payload = {k: package[k] for k in sorted(package) if k != "package_digest"}  # 计算并保存当前步骤的中间状态。
    return sha256(json.dumps(payload, sort_keys=True, separators=(",", ":")).encode()).hexdigest()  # 返回当前分支计算出的结果。

def deep_freeze46(value):  # 定义本节可复用的核心函数。
    if isinstance(value, dict):  # 按当前条件选择后续控制路径。
        return MappingProxyType({key: deep_freeze46(item) for key, item in value.items()})  # 返回当前分支计算出的结果。
    if isinstance(value, list):  # 按当前条件选择后续控制路径。
        return tuple(deep_freeze46(item) for item in value)  # 返回当前分支计算出的结果。
    return value  # 返回当前分支计算出的结果。

CONFIG46 = {"image_size": 16, "in_channels": 1, "num_classes": 3, "embed_dim": 16,  # 计算并保存当前步骤的中间状态。
            "patch_size": 2, "window_sizes": [4, 2], "shift_size": 2}  # 执行当前语句以推进本节示例。
SPLIT46 = {"generator": "torch.Generator", "train_seed": SEED + 1, "test_seed": SEED + 2,  # 计算并保存当前步骤的中间状态。
           "train_count": 36, "test_count": 18, "selection": "fixed-steps-no-test-selection"}  # 执行当前语句以推进本节示例。
PREPROCESS46 = {"range": [0.0, 1.0], "dtype": "float32", "layout": "NCHW",  # 计算并保存当前步骤的中间状态。
                "resize": None, "pad": None, "finite": True}  # 执行当前语句以推进本节示例。
COORDINATES46 = {"patch": "strict-divisible", "merge_odd": "right-bottom-zero-pad",  # 计算并保存当前步骤的中间状态。
                 "merge_order": ["even-even", "odd-even", "even-odd", "odd-odd"], "token_layout": "BHWC"}  # 执行当前语句以推进本节示例。
ATTENTION46 = {"qk_scale": "head_dim**-0.5", "relative_index": "signed-2d-row-major",  # 计算并保存当前步骤的中间状态。
               "mask_allowed": "0-or-<=-20", "published_blocker": -100.0,  # 计算并保存当前步骤的中间状态。
               "shift": "negative-roll-attend-positive-roll"}  # 执行当前语句以推进本节示例。
LABELS46 = {"0": "vertical", "1": "horizontal", "2": "cross"}  # 计算并保存当前步骤的中间状态。
TRAIN_RECIPE46 = {"seed": SEED, "optimizer": "AdamW", "lr": 5e-3, "weight_decay": 1e-4,  # 计算并保存当前步骤的中间状态。
                  "steps": 38, "objective": "cross_entropy", "controlled_fixture": True}  # 执行当前语句以推进本节示例。

published_state46 = {k: v.detach().cpu().clone() for k, v in swin_model.state_dict().items()}  # 计算并保存当前步骤的中间状态。
buffer46 = io.BytesIO(); torch.save(published_state46, buffer46)  # 计算并保存当前步骤的中间状态。
artifact46 = {  # 计算并保存当前步骤的中间状态。
    "artifact_id": "tiny-swin-patterns-v1",  # 执行当前语句以推进本节示例。
    "config": CONFIG46,  # 执行当前语句以推进本节示例。
    "state_hex": buffer46.getvalue().hex(),  # 执行当前语句以推进本节示例。
    "state_digest": state_digest46(published_state46),  # 执行当前语句以推进本节示例。
    "data_digest": tensor_digest46(train_images46, train_labels46, test_images46, test_labels46),  # 执行当前语句以推进本节示例。
    "split_recipe": SPLIT46,  # 执行当前语句以推进本节示例。
    "preprocess": PREPROCESS46,  # 执行当前语句以推进本节示例。
    "coordinate_recipe": COORDINATES46,  # 执行当前语句以推进本节示例。
    "attention_recipe": ATTENTION46,  # 执行当前语句以推进本节示例。
    "labels": LABELS46,  # 执行当前语句以推进本节示例。
    "training_recipe": TRAIN_RECIPE46,  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
artifact46["package_digest"] = package_digest46(artifact46)  # 计算并保存当前步骤的中间状态。
PUBLISHER_REGISTRY46 = MappingProxyType({artifact46["artifact_id"]: artifact46["package_digest"]})  # 计算并保存当前步骤的中间状态。

@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class PublishedSwin:  # 定义承载本节状态与行为的数据结构。
    model: TinySwinClassifier  # 执行当前语句以推进本节示例。
    config: object  # 执行当前语句以推进本节示例。
    preprocess: object  # 执行当前语句以推进本节示例。
    coordinates: object  # 执行当前语句以推进本节示例。
    attention: object  # 执行当前语句以推进本节示例。
    labels: object  # 执行当前语句以推进本节示例。
    split: object  # 执行当前语句以推进本节示例。

    def predict(self, images):  # 定义本节可复用的核心函数。
        if images.dtype != torch.float32 or not torch.isfinite(images).all():  # 按当前条件选择后续控制路径。
            raise ValueError("published Swin expects finite float32 images")  # 遇到非法合同立即显式失败。
        if images.numel() and (float(images.min()) < 0.0 or float(images.max()) > 1.0):  # 按当前条件选择后续控制路径。
            raise ValueError("published Swin expects image range [0,1]")  # 遇到非法合同立即显式失败。
        return self.model(images)  # 返回当前分支计算出的结果。

def load_published_swin46(package):  # 定义本节可复用的核心函数。
    artifact_id = package.get("artifact_id")  # 计算并保存当前步骤的中间状态。
    if artifact_id not in PUBLISHER_REGISTRY46 or package.get("package_digest") != PUBLISHER_REGISTRY46[artifact_id]:  # 按当前条件选择后续控制路径。
        raise ValueError("artifact is not approved by publisher registry")  # 遇到非法合同立即显式失败。
    if package_digest46(package) != package["package_digest"]:  # 按当前条件选择后续控制路径。
        raise ValueError("package metadata digest mismatch")  # 遇到非法合同立即显式失败。
    expected = {  # 计算并保存当前步骤的中间状态。
        "config": CONFIG46, "split_recipe": SPLIT46, "preprocess": PREPROCESS46,  # 执行当前语句以推进本节示例。
        "coordinate_recipe": COORDINATES46, "attention_recipe": ATTENTION46,  # 执行当前语句以推进本节示例。
        "labels": LABELS46, "training_recipe": TRAIN_RECIPE46,  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。
    for field, wanted in expected.items():  # 遍历输入元素以累积或检查结果。
        if package.get(field) != wanted:  # 按当前条件选择后续控制路径。
            raise ValueError(f"published semantic contract mismatch: {field}")  # 遇到非法合同立即显式失败。
    if set(package["labels"]) != {str(i) for i in range(package["config"]["num_classes"])}:  # 按当前条件选择后续控制路径。
        raise ValueError("label ids and num_classes disagree")  # 遇到非法合同立即显式失败。
    expected_data = tensor_digest46(train_images46, train_labels46, test_images46, test_labels46)  # 计算并保存当前步骤的中间状态。
    if package.get("data_digest") != expected_data:  # 按当前条件选择后续控制路径。
        raise ValueError("bound split digest mismatch")  # 遇到非法合同立即显式失败。
    state = torch.load(io.BytesIO(bytes.fromhex(package["state_hex"])), map_location="cpu", weights_only=True)  # 计算并保存当前步骤的中间状态。
    if state_digest46(state) != package["state_digest"]:  # 按当前条件选择后续控制路径。
        raise ValueError("canonical state digest mismatch")  # 遇到非法合同立即显式失败。
    cfg = package["config"]  # 计算并保存当前步骤的中间状态。
    model = TinySwinClassifier(cfg["image_size"], cfg["in_channels"], cfg["num_classes"], cfg["embed_dim"]).eval()  # 计算并保存当前步骤的中间状态。
    model.load_state_dict(state, strict=True)  # 计算并保存当前步骤的中间状态。
    return PublishedSwin(model, deep_freeze46(cfg), deep_freeze46(package["preprocess"]),  # 返回当前分支计算出的结果。
                         deep_freeze46(package["coordinate_recipe"]), deep_freeze46(package["attention_recipe"]),  # 执行当前语句以推进本节示例。
                         deep_freeze46(package["labels"]), deep_freeze46(package["split_recipe"]))  # 执行当前语句以推进本节示例。

loaded_swin46 = load_published_swin46(deepcopy(artifact46))  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    assert torch.equal(loaded_swin46.predict(test_images46[:2]), swin_model(test_images46[:2]))  # 用受控断言验证关键不变量。
assert loaded_swin46.labels["2"] == "cross"  # 用受控断言验证关键不变量。
assert loaded_swin46.coordinates["merge_odd"] == "right-bottom-zero-pad"  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    loaded_swin46.labels["0"] = "mutated"  # 计算并保存当前步骤的中间状态。
    raise AssertionError("published label map should be read-only")  # 遇到非法合同立即显式失败。
except TypeError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。

forged46 = deepcopy(artifact46)  # 计算并保存当前步骤的中间状态。
forged_state46 = torch.load(io.BytesIO(bytes.fromhex(forged46["state_hex"])), weights_only=True)  # 计算并保存当前步骤的中间状态。
forged_state46["head.bias"] = forged_state46["head.bias"] + 7  # 计算并保存当前步骤的中间状态。
forged_buffer46 = io.BytesIO(); torch.save(forged_state46, forged_buffer46)  # 计算并保存当前步骤的中间状态。
forged46["state_hex"] = forged_buffer46.getvalue().hex()  # 计算并保存当前步骤的中间状态。
forged46["state_digest"] = state_digest46(forged_state46)  # 计算并保存当前步骤的中间状态。
forged46["labels"] = {"0": "forged", "1": "forged", "2": "forged"}  # 计算并保存当前步骤的中间状态。
forged46["package_digest"] = package_digest46(forged46)  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    load_published_swin46(forged46)  # 执行当前语句以推进本节示例。
    raise AssertionError("self-rehashed replacement must fail closed")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "publisher registry" in str(exc)  # 用受控断言验证关键不变量。


## 11. 失败模式、训练/推理差异与生产差距

- **尺寸合同**：patch embedding 拒绝不可整除的输入；patch merging 明确右/下补零。动态 batching 还需把有效 token mask 一路传入 attention。
- **mask 语义**：`-100` 对 float32 足够使概率下溢；混合精度应使用与 dtype 匹配的安全负值并做数值测试。
- **复杂度**：窗口注意力降低 attention 矩阵开销，但 QKV/MLP、feature map 和 window reshape 仍消耗内存；部署要实测峰值 RSS/显存。
- **训练与推理**：本例无 dropout/drop-path；完整复现还要处理 stochastic depth、增强、EMA、AMP 和分布式随机性。
- **发布边界**：`PublishedSwin` 携带只读标签、预处理、坐标与 attention recipe；真实 registry 仍应由签名/KMS/透明日志托管。
- **质量边界**：条纹任务是受控计算图测试，不是自然图像 benchmark。上线需锁定真实 split、校准、漂移监控和回滚制品。

原始资料：

- [Swin Transformer: Hierarchical Vision Transformer using Shifted Windows](https://arxiv.org/abs/2103.14030)
- [PyTorch Conv2d 文档](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)
- [PyTorch LayerNorm 文档](https://pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html)
